# Modeling — NBA 2025-26

Fase 4 del proceso CRISP-DM. Se entrenan tres modelos para cada problema, se buscan los mejores hiperparámetros con validación cruzada temporal y se selecciona el modelo final basándose en las métricas del conjunto de test, que no se toca durante el entrenamiento.

## 1. Carga de datos procesados

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.dummy        import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble     import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline     import Pipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

sns.set_theme(style='whitegrid')

PROC = '../data/processed/'
team_train   = pd.read_csv(PROC + 'team_classification_train.csv',  parse_dates=['game_date'])
team_test    = pd.read_csv(PROC + 'team_classification_test.csv',   parse_dates=['game_date'])
player_train = pd.read_csv(PROC + 'player_regression_train.csv',   parse_dates=['game_date'])
player_test  = pd.read_csv(PROC + 'player_regression_test.csv',    parse_dates=['game_date'])

TEAM_FEATS   = [c for c in team_train.columns   if c not in ['game_id','team_id','game_date','target']]
PLAYER_FEATS = [c for c in player_train.columns if c not in ['game_id','player_id','team_id','game_date','target']]

X_tt, y_tt = team_train[TEAM_FEATS].values,     team_train['target'].values
X_te, y_te = team_test[TEAM_FEATS].values,      team_test['target'].values
X_pt, y_pt = player_train[PLAYER_FEATS].values, player_train['target'].values
X_pe, y_pe = player_test[PLAYER_FEATS].values,  player_test['target'].values

pd.DataFrame({
    'train': [team_train.shape, player_train.shape],
    'test':  [team_test.shape,  player_test.shape]
}, index=['clasificacion', 'regresion'])

## 2. Baseline

El baseline es el modelo más simple posible y define el umbral mínimo que cualquier modelo real debe superar. Para clasificación, el baseline predice siempre la clase mayoritaria. Para regresión, predice siempre el promedio histórico de puntos del jugador en el conjunto de entrenamiento. Un modelo que no supere estos baselines no aporta nada que no haga una regla trivial.

In [ ]:
clf_metrics = {}
reg_metrics = {}

tscv = TimeSeriesSplit(n_splits=5)

dc = DummyClassifier(strategy='most_frequent', random_state=42).fit(X_tt, y_tt)
dr = DummyRegressor(strategy='mean').fit(X_pt, y_pt)

yp_dc   = dc.predict(X_te)
yp_dc_p = dc.predict_proba(X_te)[:, 1]
yp_dr   = dr.predict(X_pe)

clf_metrics['Baseline'] = {
    'Accuracy': round(accuracy_score(y_te, yp_dc), 4),
    'F1':       round(f1_score(y_te, yp_dc, zero_division=0), 4),
    'AUC-ROC':  round(roc_auc_score(y_te, yp_dc_p), 4)
}
reg_metrics['Baseline'] = {
    'RMSE': round(np.sqrt(mean_squared_error(y_pe, yp_dr)), 4),
    'MAE':  round(mean_absolute_error(y_pe, yp_dr), 4),
    'R2':   round(r2_score(y_pe, yp_dr), 4)
}

pd.DataFrame({'Clasificacion': clf_metrics['Baseline'], 'Regresion': reg_metrics['Baseline']})

El baseline de clasificación alcanza un 50% de accuracy porque las clases están perfectamente balanceadas y el modelo siempre predice victoria. El AUC-ROC de 0.50 confirma que el modelo no discrimina. El baseline de regresión tiene RMSE de 8.38 puntos, equivalente a predecir siempre ~11.9 puntos independientemente del jugador o el contexto.

## 3. Problema 1 — Clasificación (resultado de partido)

**Regresión Logística**

La regresión logística modela la probabilidad de victoria como:

$$P(Y=1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots + \beta_n x_n)}}$$

Los coeficientes β representan el cambio en el log-odds de ganar por unidad estandarizada de cada feature. Es el primer modelo real porque su interpretabilidad es directa: un coeficiente positivo en `winrate_last5` significa que equipos con mayor porcentaje de victorias recientes tienen mayor probabilidad predicha de ganar.

El parámetro de regularización `C` controla el trade-off entre ajuste y complejidad: `C` pequeño impone mayor regularización (penaliza coeficientes grandes), `C` grande permite más libertad al modelo. Se busca el mejor `C` con validación cruzada temporal.

In [ ]:
pipe_lr = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr_cv   = GridSearchCV(pipe_lr, {'clf__C': [0.01, 0.1, 1.0, 10.0]}, cv=tscv, scoring='roc_auc')
lr_cv.fit(X_tt, y_tt)
lr_best = lr_cv.best_estimator_

yp_lr   = lr_best.predict(X_te)
yp_lr_p = lr_best.predict_proba(X_te)[:, 1]

clf_metrics['Log. Regression'] = {
    'Accuracy': round(accuracy_score(y_te, yp_lr), 4),
    'F1':       round(f1_score(y_te, yp_lr), 4),
    'AUC-ROC':  round(roc_auc_score(y_te, yp_lr_p), 4)
}

coef_lr = pd.Series(lr_best.named_steps['clf'].coef_[0], index=TEAM_FEATS).abs().sort_values(ascending=False)

display(pd.DataFrame({'mejor C': [lr_cv.best_params_['clf__C']], **clf_metrics['Log. Regression']}))
coef_lr.head(10).to_frame('coef. abs. estandarizado')

La regresión logística alcanza AUC-ROC de 0.727 en test, el mejor de los tres clasificadores. El coeficiente de mayor magnitud es `winrate_last5`: la racha reciente del equipo en los últimos 5 partidos es el predictor más directo. `plus_minus_last5` también tiene peso relevante, capturando el diferencial de anotación reciente. La localía (`is_home`) contribuye de forma consistente, lo que confirma el conocido efecto de cancha en la NBA.

**Random Forest Classifier**

Un árbol de decisión divide el espacio de features en regiones rectangulares mediante cortes binarios sucesivos. El problema de un solo árbol es la alta varianza: pequeños cambios en los datos de entrenamiento producen árboles muy distintos. Random Forest reduce esta varianza con dos mecanismos: (1) **bagging** — cada árbol se entrena sobre una muestra con reemplazo del conjunto de entrenamiento, y (2) **feature subsampling** — en cada nodo, la división óptima se busca solo sobre un subconjunto aleatorio de features. La predicción final es la moda de los N árboles.

Los hiperparámetros clave:
- `n_estimators`: número de árboles. Más árboles reducen varianza hasta un punto de rendimiento decreciente.
- `max_depth`: profundidad máxima de cada árbol. Árboles profundos capturan patrones complejos pero pueden memorizar ruido.
- `min_samples_split`: mínimo de muestras requeridas para dividir un nodo. Valores altos regularizan el modelo.

In [ ]:
rf_clf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {'n_estimators': [100, 200], 'max_depth': [4, 6, 8], 'min_samples_split': [10, 20, 40]},
    n_iter=12, cv=tscv, scoring='roc_auc', random_state=42, n_jobs=-1
)
rf_clf.fit(X_tt, y_tt)

yp_rf   = rf_clf.predict(X_te)
yp_rf_p = rf_clf.predict_proba(X_te)[:, 1]

clf_metrics['Random Forest'] = {
    'Accuracy': round(accuracy_score(y_te, yp_rf), 4),
    'F1':       round(f1_score(y_te, yp_rf), 4),
    'AUC-ROC':  round(roc_auc_score(y_te, yp_rf_p), 4)
}

display(pd.Series(rf_clf.best_params_).to_frame('mejores hiperparámetros'))
pd.Series(clf_metrics['Random Forest']).to_frame('Random Forest')

Random Forest obtiene la mejor accuracy (0.678) y F1 (0.694) de los tres clasificadores, pero un AUC-ROC de 0.707, inferior al de LR. Los hiperparámetros óptimos incluyen `max_depth=4` y `min_samples_split=40`, lo que indica que el problema requiere árboles poco profundos para no capturar ruido. La regularización fuerte es coherente con el tamaño del dataset de clasificación (1.626 filas de entrenamiento).

**XGBoost Classifier**

XGBoost usa **boosting** en lugar de bagging: los árboles se añaden secuencialmente. Cada árbol aprende los residuos del modelo acumulado hasta ese momento, reduciendo iterativamente el error. La diferencia con Random Forest es que en boosting los árboles no son independientes: cada uno corrige los errores del anterior.

Hiperparámetros relevantes:
- `learning_rate` (η): cuánto contribuye cada árbol al ensemble. Un η pequeño requiere más árboles pero generaliza mejor.
- `n_estimators`: número de árboles en la secuencia de boosting.
- `max_depth`: profundidad de cada árbol. XGBoost usa árboles más poco profundos que Random Forest por defecto.
- `subsample`: fracción de muestras por árbol. Equivalente al bagging de RF pero aplicado al boosting.

In [ ]:
xgb_clf = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    {'n_estimators': [100, 200], 'max_depth': [3, 4, 5],
     'learning_rate': [0.05, 0.1, 0.15], 'subsample': [0.7, 0.9]},
    n_iter=12, cv=tscv, scoring='roc_auc', random_state=42, n_jobs=-1
)
xgb_clf.fit(X_tt, y_tt)

yp_xc   = xgb_clf.predict(X_te)
yp_xc_p = xgb_clf.predict_proba(X_te)[:, 1]

clf_metrics['XGBoost'] = {
    'Accuracy': round(accuracy_score(y_te, yp_xc), 4),
    'F1':       round(f1_score(y_te, yp_xc), 4),
    'AUC-ROC':  round(roc_auc_score(y_te, yp_xc_p), 4)
}

display(pd.Series(xgb_clf.best_params_).to_frame('mejores hiperparámetros'))
pd.Series(clf_metrics['XGBoost']).to_frame('XGBoost')

XGBoost obtiene el AUC-ROC más bajo de los tres modelos (0.663). Con solo 1.626 filas de entrenamiento, el boosting gradiente no puede explotar su capacidad de modelar interacciones complejas. Al contrario: el riesgo de overfitting sobre el ruido propio del resultado deportivo supera la ganancia de complejidad adicional. Este resultado es coherente con la literatura: XGBoost brilla en datasets de decenas de miles de filas; con datasets pequeños su ventaja desaparece.

## 4. Problema 2 — Regresión (puntos por partido)

**Regresión Lineal Múltiple**

El modelo estima:

$$\text{pts} = \beta_0 + \beta_1 \cdot \text{pts\_last5} + \beta_2 \cdot \text{min\_last5} + \cdots + \varepsilon$$

Los coeficientes β representan el cambio esperado en puntos por unidad estandarizada de cada feature, manteniendo constantes el resto. Los supuestos del modelo son: linealidad de la relación, independencia de los errores, homocedasticidad (varianza constante del error) y normalidad de los residuos. Con 14.404 observaciones y features que son promedios móviles (transformaciones lineales de las observaciones originales), la relación entre features y target es razonablemente lineal.

In [ ]:
pipe_lreg = Pipeline([('sc', StandardScaler()), ('reg', LinearRegression())])
pipe_lreg.fit(X_pt, y_pt)
yp_lreg = pipe_lreg.predict(X_pe)

reg_metrics['Lin. Regression'] = {
    'RMSE': round(np.sqrt(mean_squared_error(y_pe, yp_lreg)), 4),
    'MAE':  round(mean_absolute_error(y_pe, yp_lreg), 4),
    'R2':   round(r2_score(y_pe, yp_lreg), 4)
}

coef_lreg = pd.Series(pipe_lreg.named_steps['reg'].coef_, index=PLAYER_FEATS).abs().sort_values(ascending=False)

display(pd.Series(reg_metrics['Lin. Regression']).to_frame('Lin. Regression'))
coef_lreg.to_frame('coef. abs. estandarizado')

La regresión lineal explica el 39.8% de la varianza de puntos por partido (R² = 0.398) con un MAE de 4.96 puntos. Los coeficientes más altos son `pts_last5` y `pts_last10`, la señal más directa del aporte ofensivo reciente, seguidos de `min_last5`: los minutos jugados son el principal determinante estructural del scoring. `fg_pct_last5` y `ft_pct_last5` capturan la eficiencia, que es independiente del volumen.

**Random Forest Regressor**

El ensamble de árboles en regresión funciona igual que en clasificación, pero la predicción de cada árbol es un valor numérico (media de las muestras en la hoja) y la predicción final es la media de los N árboles. Al promediar múltiples árboles independientes, el bosque reduce la varianza de la predicción sin aumentar el sesgo, lo que lo hace más estable que un árbol único.

In [ ]:
rf_reg = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {'n_estimators': [100, 200], 'max_depth': [6, 8, 10], 'min_samples_split': [10, 20, 40]},
    n_iter=10, cv=tscv, scoring='neg_root_mean_squared_error', random_state=42, n_jobs=-1
)
rf_reg.fit(X_pt, y_pt)
yp_rfr = rf_reg.predict(X_pe)

reg_metrics['Random Forest'] = {
    'RMSE': round(np.sqrt(mean_squared_error(y_pe, yp_rfr)), 4),
    'MAE':  round(mean_absolute_error(y_pe, yp_rfr), 4),
    'R2':   round(r2_score(y_pe, yp_rfr), 4)
}

display(pd.Series(rf_reg.best_params_).to_frame('mejores hiperparámetros'))
pd.Series(reg_metrics['Random Forest']).to_frame('Random Forest')

Random Forest obtiene métricas prácticamente idénticas a la regresión lineal (RMSE 6.55 vs 6.50). A pesar de su capacidad para capturar relaciones no lineales e interacciones entre features, no supera al modelo lineal. Esto indica que la estructura del problema en esta escala temporal es esencialmente lineal: el historial reciente de un jugador predice sus puntos futuros a través de una combinación lineal, no de patrones complejos.

**XGBoost Regressor**

En regresión, XGBoost minimiza una función de pérdida cuadrática (MSE) de forma iterativa. El primer árbol predice el promedio global. Cada árbol sucesivo modela los residuos del ensemble acumulado. Este mecanismo permite que XGBoost capture relaciones no lineales y efectos de interacción que la regresión lineal ignora, aunque a costa de mayor riesgo de overfitting cuando el dataset es limitado.

In [ ]:
xgb_reg = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    {'n_estimators': [100, 200], 'max_depth': [4, 5, 6],
     'learning_rate': [0.05, 0.1, 0.15], 'subsample': [0.7, 0.9]},
    n_iter=12, cv=tscv, scoring='neg_root_mean_squared_error', random_state=42, n_jobs=-1
)
xgb_reg.fit(X_pt, y_pt)
yp_xr = xgb_reg.predict(X_pe)

reg_metrics['XGBoost'] = {
    'RMSE': round(np.sqrt(mean_squared_error(y_pe, yp_xr)), 4),
    'MAE':  round(mean_absolute_error(y_pe, yp_xr), 4),
    'R2':   round(r2_score(y_pe, yp_xr), 4)
}

display(pd.Series(xgb_reg.best_params_).to_frame('mejores hiperparámetros'))
pd.Series(reg_metrics['XGBoost']).to_frame('XGBoost')

XGBoost obtiene el RMSE más alto de los tres modelos (6.58), confirmando la tendencia. La consistencia de los tres modelos alrededor del mismo umbral de RMSE (~6.5 puntos) indica que el techo predictivo con estas features está cerca del 40% de varianza explicada. La variabilidad restante (~60%) proviene de factores no medidos: estado físico exacto del jugador en ese partido, rotaciones del entrenador en tiempo real, situaciones de foul trouble, o contexto del marcador.

## 5. Tabla comparativa de modelos

In [ ]:
clf_df = pd.DataFrame(clf_metrics).T.round(4)
clf_df.index.name = 'Modelo'
display(clf_df)

In [ ]:
reg_df = pd.DataFrame(reg_metrics).T.round(4)
reg_df.index.name = 'Modelo'
reg_df

## 6. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rf_imp_clf  = pd.Series(rf_clf.best_estimator_.feature_importances_, index=TEAM_FEATS).sort_values()
xgb_imp_clf = pd.Series(xgb_clf.best_estimator_.feature_importances_, index=TEAM_FEATS).sort_values()

rf_imp_clf.tail(10).plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title('Random Forest — Clasificación (equipos)')
axes[0].set_xlabel('Importancia')

xgb_imp_clf.tail(10).plot(kind='barh', ax=axes[1], color='#DD8452')
axes[1].set_title('XGBoost — Clasificación (equipos)')
axes[1].set_xlabel('Importancia')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rf_imp_reg  = pd.Series(rf_reg.best_estimator_.feature_importances_, index=PLAYER_FEATS).sort_values()
xgb_imp_reg = pd.Series(xgb_reg.best_estimator_.feature_importances_, index=PLAYER_FEATS).sort_values()

rf_imp_reg.tail(10).plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title('Random Forest — Regresión (jugadores)')
axes[0].set_xlabel('Importancia')

xgb_imp_reg.tail(10).plot(kind='barh', ax=axes[1], color='#DD8452')
axes[1].set_title('XGBoost — Regresión (jugadores)')
axes[1].set_xlabel('Importancia')

plt.tight_layout()
plt.show()

**Clasificación:** En ambos modelos, `winrate_last5` y `winrate_last10` concentran la mayor importancia. Esto tiene sentido desde el dominio: la racha reciente resume el estado de forma del equipo mejor que cualquier estadística individual. `plus_minus_last5` y `pts_against_last5` también aparecen entre las más relevantes, capturando la calidad defensiva reciente. `is_home` contribuye de forma consistente, lo que confirma el efecto de localía documentado en la literatura de baloncesto.

**Regresión:** `pts_last5` domina la importancia en ambos modelos, seguido de `pts_last10` y `min_last5`. Esta jerarquía tiene sentido: los puntos recientes son el predictor más directo de los puntos futuros, y los minutos determinan la oportunidad de anotar. La `defense_rating` del rival aparece con importancia moderada, confirmando que el contexto del rival tiene peso real pero secundario respecto al historial del propio jugador.

## 7. Selección del mejor modelo

**Clasificación — mejor modelo: Regresión Logística (AUC-ROC = 0.727)**

Aunque Random Forest obtiene la mejor accuracy (0.678) y F1 (0.694), la regresión logística tiene el mejor AUC-ROC (0.727). El AUC-ROC es la métrica más apropiada para seleccionar el clasificador final porque evalúa la calidad de las probabilidades predichas a todos los umbrales de decisión posibles, no solo al umbral fijo de 0.5. Una probabilidad bien calibrada es más valiosa en contextos de toma de decisiones reales (análisis de rivales, gestión de estrategia) que una predicción binaria. Adicionalmente, la regresión logística es completamente interpretable: sus coeficientes cuantifican el impacto de cada feature en el log-odds de victoria.

Que el modelo más simple sea el mejor en AUC-ROC no es una sorpresa. El resultado deportivo tiene un componente de aleatoriedad inherente que los modelos complejos aprenden como si fuera señal. La regularización implícita de la regresión logística limita este efecto.

**Regresión — mejor modelo: Regresión Lineal (RMSE = 6.50, MAE = 4.96, R² = 0.398)**

La regresión lineal supera a Random Forest y XGBoost en las tres métricas. El resultado refleja que la relación entre el historial reciente de un jugador y sus puntos futuros es predominantemente lineal: los promedios móviles ya capturan la tendencia relevante sin necesidad de interacciones no lineales. Añadir complejidad no reduce el sesgo porque no existe sesgo sistemático que capturar, solo introduce varianza adicional.

Un R² de 0.40 es razonable para predicción de rendimiento deportivo individual. El 60% restante de varianza corresponde a factores no observables en el momento de la predicción: forma física del día, motivación, calidad del partido concreto, o decisiones tácticas del entrenador que no se reflejan en los game logs históricos.

## 8. Guardado final

In [ ]:
os.makedirs('../models',  exist_ok=True)
os.makedirs('../reports', exist_ok=True)

joblib.dump(lr_best,                    '../models/lr_classifier.joblib')
joblib.dump(rf_clf.best_estimator_,     '../models/rf_classifier.joblib')
joblib.dump(xgb_clf.best_estimator_,    '../models/xgb_classifier.joblib')
joblib.dump(pipe_lreg,                  '../models/lr_regressor.joblib')
joblib.dump(rf_reg.best_estimator_,     '../models/rf_regressor.joblib')
joblib.dump(xgb_reg.best_estimator_,    '../models/xgb_regressor.joblib')

joblib.dump(lr_best,   '../models/best_team_classifier.joblib')
joblib.dump(pipe_lreg, '../models/best_player_regressor.joblib')

clf_df.to_csv('../reports/clf_model_comparison.csv')
reg_df.to_csv('../reports/reg_model_comparison.csv')

pd.DataFrame({
    'archivo': [
        'best_team_classifier.joblib   (LR)',
        'best_player_regressor.joblib  (LinReg)',
        'clf_model_comparison.csv',
        'reg_model_comparison.csv'
    ],
    'ubicacion': ['models/', 'models/', 'reports/', 'reports/']
})